# Interpretabilidad y Comparación Estadística

## 20. XGBoost + LIME — Importancia de Características

Se usa LIME (Local Interpretable Model-agnostic Explanations) para analizar la importancia local de las features en predicciones individuales de XGBoost.

In [ ]:
# Entrenar XGBoost final
imp_lime = SimpleImputer(strategy="median")
X_tr_lime = imp_lime.fit_transform(X_train_r)
X_te_lime = imp_lime.transform(X_test_r)

xgb_final = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                          random_state=42, verbosity=0)
xgb_final.fit(X_tr_lime, y_train_r)

# Importancia global (feature importance nativa de XGBoost)
importances = xgb_final.feature_importances_
feat_imp = pd.DataFrame({"feature": feature_cols, "importance": importances})
feat_imp = feat_imp.sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp["feature"][::-1], feat_imp["importance"][::-1], color="#2196F3")
plt.title("Top 15 Features — XGBoost (Importancia global)")
plt.xlabel("Importancia")
plt.tight_layout()
plt.show()

# LIME: explicación local de una predicción
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_tr_lime, feature_names=feature_cols, mode="regression",
    random_state=42
)

# Explicar una predicción del test (índice 100)
idx_explain = 100
exp = explainer.explain_instance(X_te_lime[idx_explain], xgb_final.predict, num_features=10)

print(f"\nPredicción para observación {idx_explain}: {xgb_final.predict(X_te_lime[idx_explain:idx_explain+1])[0]:.6f}")
print(f"Valor real: {y_test_r.iloc[idx_explain]:.6f}")
print("\nTop 10 features (LIME):")
for feat, weight in exp.as_list():
    print(f"  {feat}: {weight:.6f}")

# Gráfico LIME
fig = exp.as_pyplot_figure()
plt.title(f"LIME — Explicación local (observación {idx_explain})")
plt.tight_layout()
plt.show()

## 21. Comparación Estadística

### 21.1 Test de Diebold-Mariano (Regresión)
Compara la precisión predictiva entre modelos basándose en los errores de predicción.

In [ ]:
def diebold_mariano(e1, e2, h=1):
    """Test de Diebold-Mariano para comparar dos series de errores de predicción."""
    d = e1**2 - e2**2
    mean_d = np.mean(d)
    var_d = np.var(d, ddof=1)
    # Corrección por autocorrelación (Newey-West con h-1 lags)
    T = len(d)
    for k in range(1, h):
        gamma_k = np.cov(d[:-k], d[k:])[0, 1]
        var_d += 2 * (1 - k/h) * gamma_k / T
    se_d = np.sqrt(var_d / T)
    dm_stat = mean_d / se_d if se_d > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return dm_stat, p_value

# Comparar todos los pares de modelos de regresión
model_names = list(preds_test_reg.keys())
dm_results = []

for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        name_a = model_names[i]
        name_b = model_names[j]
        e_a = residuals_reg[name_a]
        e_b = residuals_reg[name_b]
        dm_stat, p_val = diebold_mariano(e_a, e_b)

        dm_results.append({
            "Modelo A": name_a,
            "Modelo B": name_b,
            "DM Stat": round(dm_stat, 4),
            "p-valor": round(p_val, 4),
            "Significativo (p<0.05)": "Sí" if p_val < 0.05 else "No"
        })

print("=== Test de Diebold-Mariano (por pares) ===")
print(pd.DataFrame(dm_results).to_string(index=False))

### 21.2 Bootstrap — Intervalos de Confianza

In [ ]:
def bootstrap_metric(y_true, y_pred, metric_fn, n_boot=1000, ci=95, seed=42):
    """Bootstrap para intervalos de confianza de una métrica."""
    rng = np.random.RandomState(seed)
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    lower = np.percentile(scores, (100 - ci) / 2)
    upper = np.percentile(scores, 100 - (100 - ci) / 2)
    return np.mean(scores), lower, upper

# Bootstrap para RMSE de regresión
boot_results_reg = []
for name, yp in preds_test_reg.items():
    rmse_fn = lambda yt, yp: np.sqrt(mean_squared_error(yt, yp))
    mean_val, lo, hi = bootstrap_metric(y_test_r.values, yp, rmse_fn)
    boot_results_reg.append({
        "Modelo": name, "Métrica": "RMSE",
        "Media": round(mean_val, 6),
        "IC 95% Inferior": round(lo, 6),
        "IC 95% Superior": round(hi, 6)
    })

print("=== Bootstrap IC 95% — RMSE Regresión ===")
print(pd.DataFrame(boot_results_reg).to_string(index=False))

# Bootstrap para AUC de clasificación
boot_results_clf = []
for name, yprob in probs_clf.items():
    auc_fn = lambda yt, yp: roc_auc_score(yt, yp)
    try:
        mean_val, lo, hi = bootstrap_metric(y_test_c.values, yprob, auc_fn)
        boot_results_clf.append({
            "Modelo": name, "Métrica": "AUC",
            "Media": round(mean_val, 4),
            "IC 95% Inferior": round(lo, 4),
            "IC 95% Superior": round(hi, 4)
        })
    except:
        pass

print("\n=== Bootstrap IC 95% — AUC Clasificación ===")
print(pd.DataFrame(boot_results_clf).to_string(index=False))

### 21.3 Test de DeLong (Clasificación)

In [ ]:
def delong_test(y_true, prob_a, prob_b):
    """Test de DeLong simplificado para comparar dos AUCs."""
    from scipy.stats import norm
    auc_a = roc_auc_score(y_true, prob_a)
    auc_b = roc_auc_score(y_true, prob_b)

    # Bootstrap para estimar varianza de la diferencia
    n_boot = 2000
    rng = np.random.RandomState(42)
    diffs = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        try:
            a = roc_auc_score(y_true.values[idx], prob_a[idx])
            b = roc_auc_score(y_true.values[idx], prob_b[idx])
            diffs.append(a - b)
        except:
            continue

    se = np.std(diffs) if len(diffs) > 0 else 1e-10
    z = (auc_a - auc_b) / se if se > 0 else 0
    p_value = 2 * (1 - norm.cdf(abs(z)))

    return auc_a - auc_b, p_value

# Comparar pares de clasificadores
clf_names = list(probs_clf.keys())
delong_results = []

for i in range(len(clf_names)):
    for j in range(i+1, len(clf_names)):
        name_a = clf_names[i]
        name_b = clf_names[j]
        diff, p_val = delong_test(y_test_c, probs_clf[name_a], probs_clf[name_b])
        delong_results.append({
            "Modelo A": name_a, "Modelo B": name_b,
            "Diff AUC": round(diff, 4), "p-valor": round(p_val, 4),
            "Significativo": "Sí" if p_val < 0.05 else "No"
        })

print("=== Test de DeLong (por pares de clasificadores) ===")
print(pd.DataFrame(delong_results).to_string(index=False))